In [1]:
import os

if os.path.exists("matches_features.csv"):
    print("Fichier trouve")
else:
    print("INTROUVABLE")

Fichier trouve


## 1. Charger les donnees 
On regarde toujours ses donnees avant de les utiliser. 

In [2]:
import csv
import numpy as np
import pandas as pd

df=pd.read_csv("matches_features.csv", parse_dates=["date"])
print(df.shape)  
df.head()
df.describe()
df.info()


(32381, 14)
<class 'pandas.DataFrame'>
RangeIndex: 32381 entries, 0 to 32380
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          32381 non-null  datetime64[us]
 1   home_team     32381 non-null  str           
 2   away_team     32381 non-null  str           
 3   home_score    32381 non-null  int64         
 4   away_score    32381 non-null  int64         
 5   tournament    32381 non-null  str           
 6   elo_home_pre  32381 non-null  float64       
 7   elo_away_pre  32381 non-null  float64       
 8   elo_diff      32381 non-null  float64       
 9   form_home     32381 non-null  float64       
 10  form_away     32381 non-null  float64       
 11  form_diff     32381 non-null  float64       
 12  is_neutral    32381 non-null  int64         
 13  result        32381 non-null  int64         
dtypes: datetime64[us](1), float64(6), int64(4), str(3)
memory usage: 3.5 MB


## 2. Repartition des resultats
La base a battre: que donnerait un modele qui predit toujours "victoire domicile"

In [3]:
df["result"].value_counts(normalize=True)
# 0 = victoire domicile, 1=nul, 2=victoire exterieure

result
0    0.484729
2    0.280164
1    0.235107
Name: proportion, dtype: float64

## 3. Separer entrainement/ test par date
Le modele ne doit jamais voir le futur pendant l'entrainement.

In [4]:
feature_cols= ["elo_home_pre","elo_away_pre","elo_diff","form_home","form_away","form_diff","is_neutral"]

X=df[feature_cols]
y=df["result"]

train_mask= df["date"]< "2022-01-01" 
X_train, X_test= X[train_mask], X[~train_mask]
y_train, y_test= y[train_mask], y[~train_mask]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (27719, 7) Test: (4662, 7)


## 4. Entrainer XGBoost
La seule cellule qui "apprend"; tout le reste sert a preparer ou a juger

In [5]:
from xgboost import XGBClassifier

model= XGBClassifier(
    n_estimators=300, # nombre d'arbres
    max_depth=4,   # profondeur max par arbre
    learning_rate=0.05,  #poids de chaque correction
    subsample=0.8,  #% de lignes vues par arbre
    colsample_bytree=0.8,  # %de colonnes vues par arbre
    objective="multi:softprob",
    num_class=3,
    eval_metrics="mlogloss",
    random_state=42,
)
model.fit(X_train, y_train)
print("Entrainement termine.")

/home/rwan-alex/Machine Learning/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [21:38:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Entrainement termine.


## 5. Evaluer
Est- on meilleur que 48% (la base de l'etape 2)?

In [6]:
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

pred= model.predict(X_test)
proba= model.predict_proba(X_test)

print("Accuracy :", accuracy_score(y_test, pred))
print("Log-loss :", log_loss(y_test, proba))
print()
print(confusion_matrix(y_test, y_test))
print()
print(classification_report(y_test, pred, target_names=["Victoire domicile", "Nul", "Victoire exterieur"]))

Accuracy : 0.6021021021021021
Log-loss : 0.8821262717247009

[[2227    0    0]
 [   0 1070    0]
 [   0    0 1365]]

                    precision    recall  f1-score   support

 Victoire domicile       0.62      0.87      0.73      2227
               Nul       0.33      0.01      0.03      1070
Victoire exterieur       0.57      0.62      0.60      1365

          accuracy                           0.60      4662
         macro avg       0.51      0.50      0.45      4662
      weighted avg       0.54      0.60      0.53      4662



## 6. Quelles features comptent le plus?

In [7]:
feat_imp= pd.Series(model.feature_importances_, index=feature_cols)
feat_imp.sort_values(ascending=False)

elo_diff        0.413084
is_neutral      0.151874
elo_home_pre    0.131203
elo_away_pre    0.127420
form_diff       0.076859
form_away       0.050816
form_home       0.048744
dtype: float32

## 7. Re-entrainer sur tout l'historique
Une fois valide, on ne se prive plus de donnees pour la vraie prediction

In [8]:
final_model= XGBClassifier(
    n_estimators=300, # nombre d'arbres
    max_depth=4,   # profondeur max par arbre
    learning_rate=0.05,  #poids de chaque correction
    subsample=0.8,  #% de lignes vues par arbre
    colsample_bytree=0.8,  # %de colonnes vues par arbre
    objective="multi:softprob",
    num_class=3,
    eval_metrics="mlogloss",
    random_state=42,    
)
final_model.fit(X,y)
print("Modele final entraine sur", len(X), "matchs.")

/home/rwan-alex/Machine Learning/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [21:38:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Modele final entraine sur 32381 matchs.


## 8. Predire les vrais matchs a venir
Le moment qu'on attendait: des matchs pas encore joues.

In [9]:
fixtures= pd.read_csv("quarts_finale_a_predire.csv")
X_fixtures= fixtures[feature_cols]
proba_fixtures= final_model.predict_proba(X_fixtures)

for i, row in fixtures.iterrows():
    p=proba_fixtures[i]

    print(f"{row["home_team"]} vs {row["away_team"]} ({row["stage"]})")
    print(f" Victoire {row['home_team']}: {p[0]* 100:.1f}%")
    print(f"Nul: {p[1]*100:.1f}%")
    print(f" Victoire{row["away_team"]}: {p[2]*100:.1f}%")
    print()

Argentina vs Egypt (Huitieme de finale)
 Victoire Argentina: 75.6%
Nul: 14.3%
 VictoireEgypt: 10.1%

Switzerland vs Colombia (Huitieme de finale)
 Victoire Switzerland: 19.6%
Nul: 34.4%
 VictoireColombia: 46.0%

France vs Morocco (Quart de finale)
 Victoire France: 37.8%
Nul: 27.0%
 VictoireMorocco: 35.2%

Spain vs Belgium (Quart de finale)
 Victoire Spain: 47.3%
Nul: 27.1%
 VictoireBelgium: 25.6%

Norway vs England (Quart de finale)
 Victoire Norway: 24.0%
Nul: 26.9%
 VictoireEngland: 49.1%



In [10]:
import pandas as pd
import numpy as np

# 1. Charger les données initiales (Quarts / Huitièmes)
fixtures = pd.read_csv("quarts_finale_a_predire.csv")

# Fonction rapide pour extraire les caractéristiques d'une équipe depuis le dataset initial
def get_team_stats(team_name, df_source):
    # On cherche une ligne où l'équipe était à domicile ou à l'extérieur pour récupérer ses constantes (Elo, Forme)
    row_home = df_source[df_source['home_team'] == team_name]
    if not row_home.empty:
        return row_home.iloc[0]['elo_home_pre'], row_home.iloc[0]['form_home']
    
    row_away = df_source[df_source['away_team'] == team_name]
    if not row_away.empty:
        return row_away.iloc[0]['elo_away_pre'], row_away.iloc[0]['form_away']
    
    return 1500.0, 0.5  # Valeurs par défaut si non trouvée

# Fonction pour faire prédire un match au modèle et renvoyer le vainqueur (plus forte probabilité)
def predire_match(home_team, away_team, df_source, model, feature_cols):
    # Récupération des stats individuelles
    elo_home, form_home = get_team_stats(home_team, df_source)
    elo_away, form_away = get_team_stats(away_team, df_source)
    
    # Construction de la ligne de features attendue par XGBoost
    match_data = {
        "elo_home_pre": elo_home,
        "elo_away_pre": elo_away,
        "elo_diff": elo_home - elo_away,
        "form_home": form_home,
        "form_away": form_away,
        "form_diff": form_home - form_away,
        "is_neutral": 1  # Phase finale de Coupe du Monde = terrain neutre
    }
    
    X_match = pd.DataFrame([match_data])[feature_cols]
    proba = model.predict_proba(X_match)[0] # [P(0:Victoire Dom), P(1:Nul), P(2:Victoire Ext)]
    
    print(f"🏟️  {home_team} ({proba[0]*100:.1f}%) vs {away_team} ({proba[2]*100:.1f}%) [Nul: {proba[1]*100:.1f}%]")
    
    # En phase à élimination directe, on prend le max entre Victoire Domicile (0) et Victoire Extérieur (2)
    if proba[0] >= proba[2]:
        print(f"✅ Vainqueur qualifié : {home_team}\n")
        return home_team
    else:
        print(f"✅ Vainqueur qualifié : {away_team}\n")
        return away_team

# --- SIMULATION DU TOURNOI ---
print("--- SIMULATION DES DEMI-FINALES ---")
# Exemple basé sur tes données (A adapter selon les vrais qualifiés des Quarts !)
# Supposons que les qualifiés des quarts soient : Argentine, France, Espagne, Angleterre
demi1_home = "Argentina"
demi1_away = "France"

demi2_home = "Spain"
demi2_away = "England"

# Prédiction des demi-finales
finaliste1 = predire_match(demi1_home, demi1_away, fixtures, final_model, feature_cols)
finaliste2 = predire_match(demi2_home, demi2_away, fixtures, final_model, feature_cols)

print("--- GRAND FINALE ---")
# Match ultime pour la Coupe du Monde
champion = predire_match(finaliste1, finaliste2, fixtures, final_model, feature_cols)

print(f"🏆 LE VAINQUEUR DE LA COUPE DU MONDE EST : {champion} 🏆")

--- SIMULATION DES DEMI-FINALES ---
🏟️  Argentina (34.7%) vs France (50.1%) [Nul: 15.2%]
✅ Vainqueur qualifié : France

🏟️  Spain (43.7%) vs England (34.3%) [Nul: 22.0%]
✅ Vainqueur qualifié : Spain

--- GRAND FINALE ---
🏟️  France (24.5%) vs Spain (56.3%) [Nul: 19.2%]
✅ Vainqueur qualifié : Spain

🏆 LE VAINQUEUR DE LA COUPE DU MONDE EST : Spain 🏆


In [11]:
import pandas as pd
import numpy as np

# 1. Charger les matchs du fichier (Huitièmes et Quarts disponibles)
fixtures = pd.read_csv("quarts_finale_a_predire.csv")

# --- FONCTIONS OUTILS ---

def get_team_stats(team_name, df_source):
    """Récupère l'Elo et la Forme d'une équipe depuis le fichier initial."""
    row_home = df_source[df_source['home_team'] == team_name]
    if not row_home.empty:
        return row_home.iloc[0]['elo_home_pre'], row_home.iloc[0]['form_home']
    
    row_away = df_source[df_source['away_team'] == team_name]
    if not row_away.empty:
        return row_away.iloc[0]['elo_away_pre'], row_away.iloc[0]['form_away']
    
    return 1500.0, 0.5  # Valeurs par défaut

def predire_match(home_team, away_team, df_source, model, feature_cols):
    """Simule un match et retourne le vainqueur selon les probabilités."""
    elo_home, form_home = get_team_stats(home_team, df_source)
    elo_away, form_away = get_team_stats(away_team, df_source)
    
    # Recalcul des features pour le nouveau match
    match_data = {
        "elo_home_pre": elo_home,
        "elo_away_pre": elo_away,
        "elo_diff": elo_home - elo_away,
        "form_home": form_home,
        "form_away": form_away,
        "form_diff": form_home - form_away,
        "is_neutral": 1  # Terrain neutre pour la phase finale
    }
    
    X_match = pd.DataFrame([match_data])[feature_cols]
    proba = model.predict_proba(X_match)[0]  # [Victoire Dom, Nul, Victoire Ext]
    
    # Affichage des probabilités
    print(f"🏟️  {home_team} ({proba[0]*100:.1f}%) vs {away_team} ({proba[2]*100:.1f}%) [Nul: {proba[1]*100:.1f}%]")
    
    # En élimination directe, on qualifie celui qui a la plus forte probabilité de victoire directe
    if proba[0] >= proba[2]:
        print(f"👉 Qualifié : {home_team}\n")
        return home_team
    else:
        print(f"👉 Qualifié : {away_team}\n")
        return away_team


# --- SIMULATION AUTOMATIQUE DU TOURNOI ---

# Étape A : Jouer les matchs présents dans ton fichier CSV
print("========================================")
print("🏆 SIMULATION DES MATCHS DU FICHIER CSV 🏆")
print("========================================\n")

vainqueurs_initiaux = []

for i, row in fixtures.iterrows():
    print(f"--- {row['stage']} ---")
    X_match = pd.DataFrame([row[feature_cols]])
    proba = final_model.predict_proba(X_match)[0]
    
    print(f"🏟️  {row['home_team']} ({proba[0]*100:.1f}%) vs {row['away_team']} ({proba[2]*100:.1f}%) [Nul: {proba[1]*100:.1f}%]")
    
    if proba[0] >= proba[2]:
        print(f"👉 Qualifié : {row['home_team']}\n")
        vainqueurs_initiaux.append(row['home_team'])
    else:
        print(f"👉 Qualifié : {row['away_team']}\n")
        vainqueurs_initiaux.append(row['away_team'])

# Étape B : Constitution des Demi-finales
# Ici, on prend les vainqueurs pour créer les affiches des Demis.
# D'après ton fichier, tu as 2 huitièmes (Argentine/Égypte, Suisse/Colombie) et 3 quarts.
# Disons que les 4 équipes qui se qualifient pour les vraies demi-finales mondiales sont issues de cette liste.
# Pour l'exemple informatique, on prend les 4 premiers vainqueurs obtenus :
print("========================================")
print("🔥 SIMULATION DES DEMI-FINALES 🔥")
print("========================================\n")

# Séparation en deux affiches
demi1_home, demi1_away = vainqueurs_initiaux[0], vainqueurs_initiaux[1]
demi2_home, demi2_away = vainqueurs_initiaux[2], vainqueurs_initiaux[3]

print("Demi-finale 1 :")
finaliste1 = predire_match(demi1_home, demi1_away, fixtures, final_model, feature_cols)

print("Demi-finale 2 :")
finaliste2 = predire_match(demi2_home, demi2_away, fixtures, final_model, feature_cols)


# Étape C : La Grande Finale
print("========================================")
print("👑 LA GRANDE FINALE DE LA COUPE DU MONDE 👑")
print("========================================\n")

champion = predire_match(finaliste1, finaliste2, fixtures, final_model, feature_cols)

print("  " + "="*40)
print(f" 🎉 LE CHAMPION DU MONDE PRÉDIT EST : {champion.upper()} !!! 🎉")
print("  " + "="*40)

🏆 SIMULATION DES MATCHS DU FICHIER CSV 🏆

--- Huitieme de finale ---
🏟️  Argentina (75.6%) vs Egypt (10.1%) [Nul: 14.3%]
👉 Qualifié : Argentina

--- Huitieme de finale ---
🏟️  Switzerland (19.6%) vs Colombia (46.0%) [Nul: 34.4%]
👉 Qualifié : Colombia

--- Quart de finale ---
🏟️  France (37.8%) vs Morocco (35.2%) [Nul: 27.0%]
👉 Qualifié : France

--- Quart de finale ---
🏟️  Spain (47.3%) vs Belgium (25.6%) [Nul: 27.1%]
👉 Qualifié : Spain

--- Quart de finale ---
🏟️  Norway (24.0%) vs England (49.1%) [Nul: 26.9%]
👉 Qualifié : England

🔥 SIMULATION DES DEMI-FINALES 🔥

Demi-finale 1 :
🏟️  Argentina (43.3%) vs Colombia (33.9%) [Nul: 22.7%]
👉 Qualifié : Argentina

Demi-finale 2 :
🏟️  France (24.5%) vs Spain (56.3%) [Nul: 19.2%]
👉 Qualifié : Spain

👑 LA GRANDE FINALE DE LA COUPE DU MONDE 👑

🏟️  Argentina (27.2%) vs Spain (53.9%) [Nul: 18.9%]
👉 Qualifié : Spain

 🎉 LE CHAMPION DU MONDE PRÉDIT EST : SPAIN !!! 🎉


In [12]:
import pandas as pd
import numpy as np

# 1. Charger les matchs du fichier CSV
fixtures = pd.read_csv("quarts_finale_a_predire.csv")

# --- FONCTIONS OUTILS ---

def get_team_stats(team_name, df_source):
    """Récupère l'Elo et la Forme d'une équipe depuis le fichier initial."""
    row_home = df_source[df_source['home_team'] == team_name]
    if not row_home.empty:
        return row_home.iloc[0]['elo_home_pre'], row_home.iloc[0]['form_home']
    
    row_away = df_source[df_source['away_team'] == team_name]
    if not row_away.empty:
        return row_away.iloc[0]['elo_away_pre'], row_away.iloc[0]['form_away']
    
    return 1500.0, 0.5

def predire_match(home_team, away_team, df_source, model, feature_cols):
    """Simule un match et retourne le vainqueur selon les probabilités."""
    elo_home, form_home = get_team_stats(home_team, df_source)
    elo_away, form_away = get_team_stats(away_team, df_source)
    
    match_data = {
        "elo_home_pre": elo_home,
        "elo_away_pre": elo_away,
        "elo_diff": elo_home - elo_away,
        "form_home": form_home,
        "form_away": form_away,
        "form_diff": form_home - form_away,
        "is_neutral": 1
    }
    
    X_match = pd.DataFrame([match_data])[feature_cols]
    proba = model.predict_proba(X_match)[0]
    
    print(f"🏟️  {home_team} ({proba[0]*100:.1f}%) vs {away_team} ({proba[2]*100:.1f}%) [Nul: {proba[1]*100:.1f}%]")
    
    if proba[0] >= proba[2]:
        return home_team
    else:
        return away_team


# --- SIMULATION ET AFFICHAGE PAR ÉTAPE ---

# Listes pour suivre les qualifiés à chaque niveau
vainqueurs_huitièmes = []
vainqueurs_quarts = []

print("========================================")
print(" 🛑 ÉTAPE 1 : MATCHS DU FICHIER CSV     ")
print("========================================\n")

for i, row in fixtures.iterrows():
    stage = row['stage']
    X_match = pd.DataFrame([row[feature_cols]])
    proba = final_model.predict_proba(X_match)[0]
    
    # Détermination du gagnant
    gagnant = row['home_team'] if proba[0] >= proba[2] else row['away_team']
    
    print(f"[{stage}] {row['home_team']} vs {row['away_team']}")
    print(f"👉 Qualifié : {gagnant}\n")
    
    # Rangement par étape selon le libellé du CSV
    if "Huitieme" in stage:
        vainqueurs_huitièmes.append(gagnant)
    else:
        vainqueurs_quarts.append(gagnant)

# Affichage des bilans des premiers tours
print("--- BILAN DES PREMIERS TOURS ---")
print(f"Vainqueurs des Huitièmes détectés : {vainqueurs_huitièmes}")
print(f"Vainqueurs des Quarts détectés     : {vainqueurs_quarts}\n")


print("========================================")
print(" 🔥 ÉTAPE 2 : DEMI-FINALES             ")
print("========================================\n")

# On regroupe les qualifiés pour former les 4 demi-finalistes.
# Ici on prend les vainqueurs des quarts présents + le premier qualifié des huitièmes pour l'exemple.
tous_les_qualifies = vainqueurs_quarts + vainqueurs_huitièmes
demi_finalistes = tous_les_qualifies[:4] 

demi1_home, demi1_away = demi_finalistes[0], demi_finalistes[1]
demi2_home, demi2_away = demi_finalistes[2], demi_finalistes[3]

print("Match Demi-finale 1 :")
finaliste1 = predire_match(demi1_home, demi1_away, fixtures, final_model, feature_cols)
print(f"📢 VAINQUEUR ET QUALIFIÉ POUR LA FINALE : {finaliste1}\n")

print("Match Demi-finale 2 :")
finaliste2 = predire_match(demi2_home, demi2_away, fixtures, final_model, feature_cols)
print(f"📢 VAINQUEUR ET QUALIFIÉ POUR LA FINALE : {finaliste2}\n")


print("========================================")
print(" 👑 ÉTAPE 3 : LA GRANDE FINALE         ")
print("========================================\n")

champion = predire_match(finaliste1, finaliste2, fixtures, final_model, feature_cols)

print("="*45)
print(f" 🏆 LE VAINQUEUR DE LA COUPE DU MONDE EST : {champion.upper()} 🏆")
print("="*45)

 🛑 ÉTAPE 1 : MATCHS DU FICHIER CSV     

[Huitieme de finale] Argentina vs Egypt
👉 Qualifié : Argentina

[Huitieme de finale] Switzerland vs Colombia
👉 Qualifié : Colombia

[Quart de finale] France vs Morocco
👉 Qualifié : France

[Quart de finale] Spain vs Belgium
👉 Qualifié : Spain

[Quart de finale] Norway vs England
👉 Qualifié : England

--- BILAN DES PREMIERS TOURS ---
Vainqueurs des Huitièmes détectés : ['Argentina', 'Colombia']
Vainqueurs des Quarts détectés     : ['France', 'Spain', 'England']

 🔥 ÉTAPE 2 : DEMI-FINALES             

Match Demi-finale 1 :
🏟️  France (24.5%) vs Spain (56.3%) [Nul: 19.2%]
📢 VAINQUEUR ET QUALIFIÉ POUR LA FINALE : Spain

Match Demi-finale 2 :
🏟️  England (21.7%) vs Argentina (46.7%) [Nul: 31.6%]
📢 VAINQUEUR ET QUALIFIÉ POUR LA FINALE : Argentina

 👑 ÉTAPE 3 : LA GRANDE FINALE         

🏟️  Spain (30.3%) vs Argentina (38.8%) [Nul: 30.9%]
 🏆 LE VAINQUEUR DE LA COUPE DU MONDE EST : ARGENTINA 🏆
